# Churn Risk Scoring — Business Analysis

This notebook translates the findings from `01_eda.ipynb` into economic terms.
The goal is to quantify the business impact of churn and estimate the value
of a retention strategy — framing the model not as an ML exercise but as a
revenue protection tool.

## 1. Assumptions & Data Limitations

### Calculated from data
- **ARPU (Average Revenue Per User)** — derived from `MonthlyCharges` mean. Reflects the actual revenue mix of this customer base.
- **Churn rate** — 26.5% of customers churned in the observation period.

### External assumptions
- **Retention cost: AUD $15/customer** — estimated cost of a retention action in the
  Australian telco market (outbound call, targeted discount, or bundled offer).
  Source: industry approximation based on Telstra/Optus public reporting.

- **Average customer lifetime: 24 months** — assumed based on industry benchmarks.
  
  > **Note on censored data:** Ideally this would be estimated from the dataset using
  > survival analysis (e.g. Kaplan-Meier). However, `tenure` in this dataset includes
  > customers who have *not yet churned* — we don't know their true final lifetime.
  > Using the raw mean of tenure would underestimate the real value. The lifetime of
  > churned customers specifically averages ~18 months in this dataset, but this too
  > is a lower bound. We use **24 months** as a conservative industry-aligned estimate
  > and acknowledge this as an assumption, not a derived fact.

## 2. Revenue Impact

In [2]:
import pandas as pd
import numpy as np

import kagglehub
path = kagglehub.dataset_download("blastchar/telco-customer-churn")
df = pd.read_csv(f"{path}/WA_Fn-UseC_-Telco-Customer-Churn.csv")

/Users/robertogarces/miniforge3/envs/churn-risk-scoring/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Fix data types (same as EDA)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [4]:
# --- Calculated from data ---
ARPU_AUD = df['MonthlyCharges'].mean()
churn_rate = df['Churn'].mean()
n_churned = df['Churn'].sum()
n_customers = len(df)

In [5]:
# --- External assumptions ---
RETENTION_COST_AUD = 15.0
AVG_LIFETIME_MONTHS = 24

In [6]:
# --- Derived metrics ---
monthly_revenue_at_risk = n_churned * ARPU_AUD
clv_at_risk = n_churned * ARPU_AUD * AVG_LIFETIME_MONTHS
total_retention_cost = n_churned * RETENTION_COST_AUD

In [7]:
print("=" * 55)
print("   REVENUE IMPACT SUMMARY")
print("=" * 55)
print(f"   Total customers:              {n_customers:,}")
print(f"   Churned customers:            {n_churned:,} ({churn_rate:.1%})")
print(f"   ARPU (AUD/month):             ${ARPU_AUD:.2f}")
print(f"   Avg customer lifetime:        {AVG_LIFETIME_MONTHS} months")
print("-" * 55)
print(f"   Monthly revenue at risk:      ${monthly_revenue_at_risk:,.0f} AUD")
print(f"   Lifetime value at risk (CLV): ${clv_at_risk:,.0f} AUD")
print(f"   Cost to retain all at-risk:   ${total_retention_cost:,.0f} AUD")
print("=" * 55)

   REVENUE IMPACT SUMMARY
   Total customers:              7,043
   Churned customers:            1,869 (26.5%)
   ARPU (AUD/month):             $64.76
   Avg customer lifetime:        24 months
-------------------------------------------------------
   Monthly revenue at risk:      $121,040 AUD
   Lifetime value at risk (CLV): $2,904,950 AUD
   Cost to retain all at-risk:   $28,035 AUD


## 3. Risk Segmentation by Contract Type

Not all churned customers are equal. We segment by contract type to identify
where retention spend has the highest ROI.

In [8]:
segment = df.groupby('Contract').agg(
    n_customers=('Churn', 'count'),
    n_churned=('Churn', 'sum'),
    churn_rate=('Churn', 'mean'),
    avg_monthly_charges=('MonthlyCharges', 'mean')
).reset_index()

segment['monthly_revenue_at_risk'] = segment['n_churned'] * segment['avg_monthly_charges']
segment['clv_at_risk'] = segment['monthly_revenue_at_risk'] * AVG_LIFETIME_MONTHS
segment['retention_cost'] = segment['n_churned'] * RETENTION_COST_AUD
segment['roi_if_50pct_saved'] = (segment['monthly_revenue_at_risk'] * 0.5) / segment['retention_cost']

segment['churn_rate'] = segment['churn_rate'].map('{:.1%}'.format)
segment['avg_monthly_charges'] = segment['avg_monthly_charges'].map('${:.2f}'.format)
segment['monthly_revenue_at_risk'] = segment['monthly_revenue_at_risk'].map('${:,.0f}'.format)
segment['clv_at_risk'] = segment['clv_at_risk'].map('${:,.0f}'.format)
segment['retention_cost'] = segment['retention_cost'].map('${:,.0f}'.format)
segment['roi_if_50pct_saved'] = segment['roi_if_50pct_saved'].map('{:.1f}x'.format)

segment.set_index('Contract', inplace=True)
segment.columns = ['Customers', 'Churned', 'Churn Rate', 
                   'Avg Monthly Charges', 'Monthly Revenue at Risk',
                   'CLV at Risk', 'Retention Cost', 'ROI (50% saved)']
segment

,Customers,Churned,Churn Rate,Avg Monthly Charges,Monthly Revenue at Risk,CLV at Risk,Retention Cost,ROI (50% saved)
Contract,,,,,,,,
Month-to-month,3875,1655,42.7%,$66.40,"$109,890","$2,637,348","$24,825",2.2x
One year,1473,166,11.3%,$65.05,"$10,798","$259,154","$2,490",2.2x
Two year,1695,48,2.8%,$60.77,"$2,917","$70,008",$720,2.0x


## 4. High-Value Customer Analysis

ROI across contract types is similar (~2x), meaning retention spend is equally
efficient regardless of segment. The priority is therefore **volume** — and
month-to-month customers represent 88% of total churned revenue at risk.

A second dimension worth exploring: are high-spend customers churning more?
If so, the revenue impact is disproportionate to headcount.

In [9]:
# Segment churned customers by MonthlyCharges quartile
churned = df[df['Churn'] == 1].copy()
churned['spend_quartile'] = pd.qcut(churned['MonthlyCharges'], q=4, 
                                     labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4 (highest)'])

quartile_analysis = churned.groupby('spend_quartile', observed=True).agg(
    n_churned=('Churn', 'count'),
    avg_monthly_charges=('MonthlyCharges', 'mean'),
    avg_tenure=('tenure', 'mean')
).reset_index()

quartile_analysis['monthly_revenue_lost'] = (
    quartile_analysis['n_churned'] * quartile_analysis['avg_monthly_charges']
)
quartile_analysis['pct_of_total_revenue_lost'] = (
    quartile_analysis['monthly_revenue_lost'] / monthly_revenue_at_risk * 100
)

quartile_analysis['avg_monthly_charges'] = quartile_analysis['avg_monthly_charges'].map('${:.2f}'.format)
quartile_analysis['avg_tenure'] = quartile_analysis['avg_tenure'].map('{:.1f} months'.format)
quartile_analysis['monthly_revenue_lost'] = quartile_analysis['monthly_revenue_lost'].map('${:,.0f}'.format)
quartile_analysis['pct_of_total_revenue_lost'] = quartile_analysis['pct_of_total_revenue_lost'].map('{:.1f}%'.format)

quartile_analysis.set_index('spend_quartile', inplace=True)
quartile_analysis.columns = ['Churned Customers', 'Avg Monthly Charges', 
                              'Avg Tenure', 'Monthly Revenue Lost', '% of Total Revenue Lost']
quartile_analysis

,Churned Customers,Avg Monthly Charges,Avg Tenure,Monthly Revenue Lost,% of Total Revenue Lost
spend_quartile,,,,,
Q1 (lowest),468,$38.03,10.1 months,"$17,799",14.7%
Q2,471,$72.33,11.7 months,"$34,066",28.1%
Q3,463,$86.32,17.2 months,"$39,965",33.0%
Q4 (highest),467,$101.29,33.0 months,"$47,301",39.1%


## 5. Retention Priority Framework

Combining contract type and spend quartile, we can define a simple prioritisation
framework for retention actions — before the model exists.

The highest-ROI intervention targets customers who are:
- On a month-to-month contract (highest churn rate)
- In Q3 or Q4 spend (highest revenue at risk)
- With tenure > 12 months (deliberate decision to leave, not early dropout)

In [10]:
high_priority = df[
    (df['Contract'] == 'Month-to-month') &
    (df['MonthlyCharges'] >= df['MonthlyCharges'].quantile(0.5)) &
    (df['tenure'] >= 12)
].copy()

n_high_priority = len(high_priority)
n_high_priority_churned = high_priority['Churn'].sum()
churn_rate_high_priority = high_priority['Churn'].mean()
revenue_at_risk_high_priority = n_high_priority_churned * high_priority['MonthlyCharges'].mean()
roi_high_priority = (revenue_at_risk_high_priority * 0.5) / (n_high_priority_churned * RETENTION_COST_AUD)

print("=" * 55)
print("   HIGH-PRIORITY RETENTION SEGMENT")
print("=" * 55)
print(f"   Criteria:")
print(f"   - Contract:  Month-to-month")
print(f"   - Spend:     Above median MonthlyCharges")
print(f"   - Tenure:    >= 12 months")
print("-" * 55)
print(f"   Customers in segment:         {n_high_priority:,}")
print(f"   Churned in segment:           {n_high_priority_churned:,} ({churn_rate_high_priority:.1%})")
print(f"   Monthly revenue at risk:      ${revenue_at_risk_high_priority:,.0f} AUD")
print(f"   Retention cost (all):         ${n_high_priority_churned * RETENTION_COST_AUD:,.0f} AUD")
print(f"   ROI (50% retention success):  {roi_high_priority:.1f}x")
print("=" * 55)

   HIGH-PRIORITY RETENTION SEGMENT
   Criteria:
   - Contract:  Month-to-month
   - Spend:     Above median MonthlyCharges
   - Tenure:    >= 12 months
-------------------------------------------------------
   Customers in segment:         1,275
   Churned in segment:           536 (42.0%)
   Monthly revenue at risk:      $48,574 AUD
   Retention cost (all):         $8,040 AUD
   ROI (50% retention success):  3.0x


## 6. Summary

In [11]:
print("=" * 55)
print("   BUSINESS ANALYSIS SUMMARY")
print("=" * 55)
print(f"   Total monthly revenue at risk:   ${monthly_revenue_at_risk:,.0f} AUD")
print(f"   Total CLV at risk:               ${clv_at_risk:,.0f} AUD")
print(f"   Cost to retain all at-risk:      ${total_retention_cost:,.0f} AUD")
print("-" * 55)
print(f"   Highest risk segment:            Month-to-month")
print(f"   Revenue at risk (segment):       $109,890 AUD")
print(f"   ROI if 50% retained:             2.2x")
print("-" * 55)
print(f"   High-priority target segment:")
print(f"   (MTM + above median + tenure>12)")
print(f"   Customers:                       1,275")
print(f"   Churned:                         536 (42.0%)")
print(f"   Revenue at risk:                 $48,574 AUD")
print(f"   Retention cost:                  $8,040 AUD")
print(f"   ROI if 50% retained:             3.0x")
print("=" * 55)

   BUSINESS ANALYSIS SUMMARY
   Total monthly revenue at risk:   $121,040 AUD
   Total CLV at risk:               $2,904,950 AUD
   Cost to retain all at-risk:      $28,035 AUD
-------------------------------------------------------
   Highest risk segment:            Month-to-month
   Revenue at risk (segment):       $109,890 AUD
   ROI if 50% retained:             2.2x
-------------------------------------------------------
   High-priority target segment:
   (MTM + above median + tenure>12)
   Customers:                       1,275
   Churned:                         536 (42.0%)
   Revenue at risk:                 $48,574 AUD
   Retention cost:                  $8,040 AUD
   ROI if 50% retained:             3.0x


## 7. Key Takeaways

- A churn model is not just an ML problem — it is a **revenue protection tool**.
  The numbers above justify the investment in a predictive system.
- **Month-to-month contracts** drive 88% of churned revenue. The single highest-impact
  retention lever is incentivising contract upgrades.
- **High-spend, tenured customers** churning is the most commercially dangerous signal.
  These are deliberate decisions, not early dropouts — they require a different
  intervention strategy than new customer churn.
- Without a model, the operator cannot identify at-risk customers **before** they leave.
  With a model, retention spend can be directed at the right customers at the right time.

> Next step: build a churn scoring model that assigns a risk probability to every
> active customer — enabling proactive, targeted retention actions.